# Snippet & Lexicon-Based Sentiment Analysis

This notebook implements a lexicon-based snippet approach for ESG sentiment analysis.
It generates rule-based sentiment scores using ESG-specific keywords, negation handling,
and intensifier detection. The outputs are saved for later hybrid fusion with transformer results.

In [1]:
# 1. Setup
import json
import pandas as pd
df = pd.read_csv("../data/processed/distilbert_baseline_5class.csv")

def lexicon_to_5class(score, high=200, low=50):
    if score >= high:
        return "VERY POSITIVE"
    elif score >= low:
        return "POSITIVE"
    elif score <= -high:
        return "VERY NEGATIVE"
    elif score <= -low:
        return "NEGATIVE"
    else:
        return "NEUTRAL"



# Load lemmatized corpus (created in preprocessing)
with open("../data/cleaned_v2/preprocessed_with_lemma.jsonl", "r") as f:
    corpus = [json.loads(line) for line in f]

print(f"✅ Loaded lemmatized corpus | docs: {len(corpus)}")

✅ Loaded lemmatized corpus | docs: 9


In [2]:
# 2. Define ESG Lexicon
ESG_LEXICON = {
    "positive": [
        "sustainable", "renewable", "green", "inclusive", "responsible",
        "net", "zero", "diversity", "environmental", "governance", "social",
        "ethical", "recycling", "efficiency", "compliance", "innovation",
        "equity", "fairness", "biodiversity", "community", "wellbeing"
    ],
    "negative": [
        "emission", "emissions", "pollution", "scandal", "deforestation",
        "fine", "controversy", "risk", "hazard", "lawsuit", "waste",
        "shortage", "violation", "fraud", "breach", "exploitation",
        "child", "forced", "toxic", "unethical"
    ]
}

NEGATIONS = ["no", "not", "never", "none", "without"]
INTENSIFIERS = ["very", "highly", "extremely", "significantly"]

print("✅ Lexicon, negations, and intensifiers defined")

✅ Lexicon, negations, and intensifiers defined


In [3]:
# 3. Scoring Function
def score_tokens(tokens):
    pos, neg = 0, 0
    for i, token in enumerate(tokens):
        # Positive / negative hits
        if token in ESG_LEXICON["positive"]:
            multiplier = 2 if (i > 0 and tokens[i-1] in INTENSIFIERS) else 1
            if i > 0 and tokens[i-1] in NEGATIONS:
                neg += 1 * multiplier
            else:
                pos += 1 * multiplier
        elif token in ESG_LEXICON["negative"]:
            multiplier = 2 if (i > 0 and tokens[i-1] in INTENSIFIERS) else 1
            if i > 0 and tokens[i-1] in NEGATIONS:
                pos += 1 * multiplier
            else:
                neg += 1 * multiplier
    return {"pos": pos, "neg": neg, "score": pos - neg}

In [4]:
# Add 5-class mapping
df['lexicon_5class'] = df['score'].apply(lambda x: lexicon_to_5class(x))

# Save new CSV with 5-class column
df.to_csv("../data/processed/snippet_scores_5class.csv", index=False)
print("✅ Saved with 5-class column")
print(df['lexicon_5class'].value_counts().to_string())

✅ Saved with 5-class column
lexicon_5class
NEUTRAL    14166
